In [ ]:
!pip install keybert sentence-transformers nltk rake-nltk scikit-learn pandas matplotlib spacy
!python -m spacy download en_core_web_sm

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict
import re

# KeyBERT for keyword extraction
from keybert import KeyBERT

# NLP
import spacy
import nltk
from rake_nltk import Rake
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [ ]:

INPUT_JSON = "librispeech_local_1500.json"  

OUTPUT_KEYWORDS_JSON = "phase2_keywords.json"
OUTPUT_ANALYSIS_JSON = "phase2_analysis.json"


In [ ]:

with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    phase1_data = json.load(f)

metadata = phase1_data['metadata']
results = phase1_data['results']

# transcriptions and reference texts
transcriptions = [r['transcription'] for r in results if r['success']]
reference_texts = [r['reference_text'] for r in results if r['success']]


In [ ]:
#key_word extraction using KeyBERT
kw_model = KeyBERT()

sample_size = len(transcriptions)
sample_texts = transcriptions[:sample_size]


keybert_results = []
for i, text in enumerate(sample_texts):
    if len(text) < 10:  #skip short texts
        continue
    
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        top_n=5  # get top 5 keywords
    )
    
    keybert_results.append({
        'index': i,
        'text': text[:100] + '...',
        'keywords': keywords
    })
    
 
keybert_results[0]


In [ ]:
nlp = spacy.load("en_core_web_sm")

# NER
all_entities = defaultdict(list)

for i, text in enumerate(sample_texts):
    doc = nlp(text)
    
    for ent in doc.ents:
        all_entities[ent.label_].append(ent.text)
    

# count all entity types
entity_counts = {label: len(entities) for label, entities in all_entities.items()}
for label, count in sorted(entity_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{label:15s}: {count:4d}")

len(all_entities)

In [ ]:
question_words = {
    'who': [],
    'what': [],
    'when': [],
    'where': [],
    'why': [],
    'how': [],
    'which': [],
    'whose': []
}

# sentence that include question words
for text in transcriptions:
    text_lower = text.lower()
    for qword in question_words.keys():
        if qword in text_lower:
            #split into sentences and check if qword in the sentence
            sentences = text.split('.')
            for sent in sentences:
                if qword in sent.lower():
                    question_words[qword].append(sent.strip()[:100])

for qword, examples in question_words.items():
    print(f"{qword.upper():8s}: {len(examples):4d} ")


In [ ]:

#save keyword results
keywords_output = {
    'metadata': {
        'source_file': INPUT_JSON,
        'sample_size': sample_size,
        'total_texts': len(transcriptions)
    },
    'keybert_keywords': keybert_results
}

with open(OUTPUT_KEYWORDS_JSON, 'w', encoding='utf-8') as f:
    json.dump(keywords_output, f, ensure_ascii=False, indent=2)



#save NER and question words output
analysis_output = {
    'metadata': {
        'source_file': INPUT_JSON,
        'total_samples': len(transcriptions),
        'sample_analyzed': sample_size
    },
    'entities': {
        label: {
            'count': len(entities),
            'examples': list(set(entities))[:10]
        }
        for label, entities in all_entities.items()
    },
    'question_words': {
        qword: {
            'count': len(examples),
            'examples': examples[:5]
        }
        for qword, examples in question_words.items()
    }
}

with open(OUTPUT_ANALYSIS_JSON, 'w', encoding='utf-8') as f:
    json.dump(analysis_output, f, ensure_ascii=False, indent=2)


In [ ]:
# define question patterns based on NER
qa_patterns = {
    'PERSON': {
        'question_type': 'Who',
        'examples': list(set(all_entities.get('PERSON', [])))[:3],
        'description': 'Questions about people, characters, authors'
    },
    'DATE': {
        'question_type': 'When',
        'examples': list(set(all_entities.get('DATE', [])))[:3],
        'description': 'Questions about time, dates, periods'
    },
    'GPE': {
        'question_type': 'Where',
        'examples': list(set(all_entities.get('GPE', [])))[:3],
        'description': 'Questions about locations, places, countries'
    },
    'ORG': {
        'question_type': 'What/Which',
        'examples': list(set(all_entities.get('ORG', [])))[:3],
        'description': 'Questions about organizations, companies'
    }
}


for entity_type, pattern in qa_patterns.items():
    if pattern['examples']:
        print(f"\n{pattern['question_type']} questions")
        print(f"{entity_type}")
        print(f"{pattern['description']}")
        print(f"Answer")
        for ex in pattern['examples']:
            print(f"    - {ex}")